In [1]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

Processing /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
autograd is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=9dab7875d890d8a75cbfa5de93f4106d9b6ca92a2603118ff81ed30aa407dcb1
  Stored in directory: /root/.cache/pip/wheels/6b/b5/e0/4c79e15c0b5f2c15ecf613c720bb20daab20a666eb67135155
Successfully built autograd-gamma
Processing /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl


In [2]:
import pandas as pd
import pandas.api.types
import numpy as np
from lifelines.utils import concordance_index

class ParticipantVisibleError(Exception):
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:

    del solution[row_id_column_name]
    del submission[row_id_column_name]
    
    event_label = 'efs'
    interval_label = 'efs_time'
    prediction_label = 'prediction'
    for col in submission.columns:
        if not pandas.api.types.is_numeric_dtype(submission[col]):
            raise ParticipantVisibleError(f'Submission column {col} must be a number')
    # Merging solution and submission dfs on ID
    merged_df = pd.concat([solution, submission], axis=1)
    merged_df.reset_index(inplace=True)
    merged_df_race_dict = dict(merged_df.groupby(['race_group']).groups)
    metric_list = []
    for race in merged_df_race_dict.keys():
        # Retrieving values from y_test based on index
        indices = sorted(merged_df_race_dict[race])
        merged_df_race = merged_df.iloc[indices]
        # Calculate the concordance index
        c_index_race = concordance_index(
                        merged_df_race[interval_label],
                        -merged_df_race[prediction_label],
                        merged_df_race[event_label])
        metric_list.append(c_index_race)
    return float(np.mean(metric_list)-np.sqrt(np.var(metric_list)))


In [3]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats
import tensorflow_decision_forests as tfdf

In [4]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import h2o
from h2o.automl import H2OAutoML

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
from sklearn import tree
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [5]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [6]:
from lifelines import KaplanMeierFitter

def calculate_survival_probabilities(df, time_col, event_col):
    kmf = KaplanMeierFitter()
    kmf.fit(df[time_col], df[event_col])
    return kmf.survival_function_at_times(df[time_col]).values

def preprocess_survival_data(df, time_col='efs_time', event_col='efs'):
    df['target'] = calculate_survival_probabilities(df, time_col, event_col)
    df.loc[df[event_col] == 0, 'target'] -= 0.2  # Adjust for censored data
    return df

train = preprocess_survival_data(train)

In [7]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [8]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        if i[0]!='ID' and i[0]!='efs' and i[0]!='efs_time' and i[0]!='target':
            train[i[0]] = train[i[0]].fillna(0)
            train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))

for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        if i[0]!='ID' and i[0]!='efs' and i[0]!='efs_time' and i[0]!='target':
            test[i[0]] = test[i[0]].fillna(0)
            test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [9]:
train_x = train.drop(columns=['efs', 'efs_time', 'target'])
train_x = train_x[['dri_score', 'hla_high_res_8', 'renal_issue', 'prim_disease_hct', 'hla_match_dqb1_high', 'hla_nmdp_6', 'hla_match_c_low', 'rituximab',
 'hla_match_dqb1_low', 'cyto_score_detail', 'conditioning_intensity', 'ethnicity', 'year_hct', 'in_vivo_tcd', 'hla_match_a_high',
 'hepatic_severe', 'peptic_ulcer', 'hla_match_a_low', 'gvhd_proph', 'rheum_issue', 'sex_match', 'hla_match_b_high', 'karnofsky_score', 
 'melphalan_dose', 'hla_low_res_8']]
train_y = train['target']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.001, random_state=100)

In [11]:
x,y=list(train_x.columns),'target'

In [12]:
catboost = CatBoostRegressor(grow_policy='Depthwise',
                             iterations=1000,
                             eval_metric='RMSE',
                             random_state=0,
                             verbose=100)



cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)
    
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_test, y_test, cat_features=cat_features)
        
catboost.fit(train_pool, eval_set=(val_pool), verbose=50, early_stopping_rounds=100)

Learning rate set to 0.086288
0:	learn: 0.2581941	test: 0.2303745	best: 0.2303745 (0)	total: 64.1ms	remaining: 1m 4s
50:	learn: 0.2361398	test: 0.2127474	best: 0.2106107 (29)	total: 520ms	remaining: 9.68s
100:	learn: 0.2312136	test: 0.2166901	best: 0.2106107 (29)	total: 963ms	remaining: 8.57s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2106106654
bestIteration = 29

Shrink model to first 30 iterations.


In [13]:
cat_pre_test=catboost.predict(train_x)
#print(f'Concordance index (lifelines): {ci_lifelines(y_test, catboost.predict(X_test))}')

In [14]:
#Concordance index (lifelines): 0.6434623996950867

In [15]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv')['ID']
test_predictions = (cat_pre_test.reshape(-1))
test_predictions

array([0.3670677 , 0.51614502, 0.35255767, ..., 0.55238475, 0.44566749,
       0.37276898])

In [16]:
submission = pandas.DataFrame({'ID': id.values, 'prediction': test_predictions,})
submission

,ID,prediction
0,0,0.367068
1,1,0.516145
2,2,0.352558
3,3,0.656606
4,4,0.554028
...,...,...
28795,28795,0.530571
28796,28796,0.649507
28797,28797,0.552385
28798,28798,0.445667


In [17]:
y_true=train[['efs', 'efs_time', 'race_group']]
y_true['ID']=id

In [18]:
score(y_true.copy(), submission.copy(), 'ID') 

0.6525811111664694

In [19]:
submission.to_csv('submission.csv', index = False)